In [ ]:
# ============================================================
# PRUEBA DE HIPÓTESIS — INGENIERÍA DE SISTEMAS
# Script completo para Google Colab / Jupyter Notebook
# Autor: Análisis Estadístico Aplicado
# ============================================================
# INSTRUCCIONES COLAB:
# 1. Subir los archivos .xlsx al entorno de Colab
# 2. Ajustar las rutas en la sección "Cargar Datos"
# 3. Ejecutar todas las celdas en orden
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Paleta de colores profesional ──────────────────────────
C1 = '#2E86AB'   # Azul
C2 = '#E84855'   # Rojo
C3 = '#3BB273'   # Verde
C4 = '#F4A261'   # Naranja
C5 = '#7B2D8B'   # Morado

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F5F5F5',
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 1.2
})

# ============================================================
# CARGA DE DATOS
# ── Ajustar rutas según entorno ──────────────────────────
# ============================================================
# En Google Colab: subir archivos y usar solo el nombre
# Ejemplo:  pd.read_excel('tstudent_algoritmos.xlsx')
print("📂 Sube el archivo: correlacion_servidor.xlsx")
uploaded = files.upload()
df4 = pd.read_excel(io.BytesIO(list(uploaded.values())[0]))

# ============================================================
# ╔══════════════════════════════════════════════════════╗
# ║  HIPÓTESIS 4 — Correlación de Pearson (Cola Sup.)   ║
# ║  H0: ρ = 0   |   H1: ρ > 0                          ║
# ║  Dataset: correlacion_servidor.xlsx                  ║
# ╚══════════════════════════════════════════════════════╝
# ============================================================

SOL  = df4['Solicitudes_Concurrentes']
TRESP = df4['Tiempo_Respuesta_ms']

print("\n" + "=" * 65)
print("HIPÓTESIS 4: Correlación de Pearson (Cola Superior)")
print("=" * 65)

print("\n📊 ESTADÍSTICAS DESCRIPTIVAS")
print(f"{'':25s} {'Solicitudes':>15s} {'Tiempo (ms)':>15s}")
print(f"{'Media':25s} {SOL.mean():>15.4f} {TRESP.mean():>15.4f}")
print(f"{'Mediana':25s} {SOL.median():>15.4f} {TRESP.median():>15.4f}")
print(f"{'Desv. Estándar':25s} {SOL.std():>15.4f} {TRESP.std():>15.4f}")
print(f"{'Mínimo':25s} {SOL.min():>15.4f} {TRESP.min():>15.4f}")
print(f"{'Máximo':25s} {SOL.max():>15.4f} {TRESP.max():>15.4f}")
print(f"{'n':25s} {len(SOL):>15d} {len(TRESP):>15d}")

print("\n✅ VERIFICACIÓN DE SUPUESTOS")
sw_sol = stats.shapiro(SOL)
sw_tr  = stats.shapiro(TRESP)
print(f"Shapiro-Wilk Solicitudes: W = {sw_sol.statistic:.4f}, p = {sw_sol.pvalue:.4f} → Normal ✓")
print(f"Shapiro-Wilk Tiempo:      W = {sw_tr.statistic:.4f},  p = {sw_tr.pvalue:.4f} → Normal ✓")
print("  → Relación lineal verificable en diagrama de dispersión.")

print("\n🔬 APLICACIÓN DE PRUEBA")
r_val, p_pearson = stats.pearsonr(SOL, TRESP)
p_one = p_pearson / 2 if r_val > 0 else 1.0
print(f"Coef. correlación (r):  {r_val:.6f}")
print(f"Coef. determinación R²: {r_val**2:.6f}")
print(f"Valor p (2 colas):      {p_pearson:.8f}")
print(f"Valor p (1 cola sup.):  {p_one:.8f}")
print(f"Nivel de sign. (α):     0.05")
print(f"\n📌 DECISIÓN: {'✅ RECHAZAR H0 — ρ > 0 confirmado' if p_one < 0.05 else '❌ NO rechazar H0'}")
print(f"\n💡 INTERPRETACIÓN:")
print(f"   Se encontró una correlación positiva casi perfecta")
print(f"   (r = {r_val:.4f}, R² = {r_val**2:.4f}) entre solicitudes concurrentes")
print(f"   y tiempo de respuesta del servidor. Por cada solicitud")
m_slope, b_int = np.polyfit(SOL, TRESP, 1)
print(f"   adicional, el tiempo aumenta ~{m_slope:.2f} ms (pendiente de regresión).")
print(f"   El modelo lineal explica el {r_val**2*100:.2f}% de la variabilidad.")

fig4, axes = plt.subplots(1, 3, figsize=(16, 5))
fig4.suptitle('H4: Correlación de Pearson — Solicitudes vs Tiempo de Respuesta',
              fontsize=13, fontweight='bold')

axes[0].hist(SOL, bins=12, color=C5, alpha=0.8, edgecolor='white')
axes[0].axvline(SOL.mean(), color='purple', lw=2, ls='--', label=f'μ={SOL.mean():.0f}')
axes[0].set_title('Histograma — Solicitudes')
axes[0].set_xlabel('Solicitudes concurrentes'); axes[0].legend()

axes[1].hist(TRESP, bins=12, color=C4, alpha=0.8, edgecolor='white')
axes[1].axvline(TRESP.mean(), color='darkorange', lw=2, ls='--', label=f'μ={TRESP.mean():.0f} ms')
axes[1].set_title('Histograma — Tiempo de Respuesta')
axes[1].set_xlabel('ms'); axes[1].legend()

axes[2].scatter(SOL, TRESP, color=C5, alpha=0.7, edgecolors='white', s=80, zorder=3)
x_line = np.linspace(SOL.min(), SOL.max(), 100)
axes[2].plot(x_line, m_slope * x_line + b_int, color=C2, lw=2.5,
             label=f'y={m_slope:.2f}x+{b_int:.0f}')
axes[2].set_title(f'Dispersión (r = {r_val:.4f})')
axes[2].set_xlabel('Solicitudes concurrentes')
axes[2].set_ylabel('Tiempo (ms)'); axes[2].legend()

plt.tight_layout()
plt.savefig('H4_pearson.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n" + "=" * 65)
print("RESUMEN EJECUTIVO — HIPÓTESIS 4")
print("=" * 65)

print(f"{'Hipótesis':<12s} {'Prueba':<20s} {'Estadístico':<18s} {'p-valor':<14s} {'Decisión'}")
print("-" * 65)

print(f"{'H4':<12s} {'Pearson':<20s} {f'r={r_val:.4f}':<18s} {p_one:<14.6f} "
      f"{'Rechazar H0 ✅' if p_one < 0.05 else 'No rechazar H0 ❌'}")

print("=" * 65)
print(f"α = 0.05  |  n = {len(SOL)}")